# 04 — Model Experimentation & Multi-Model Ensembling
## Random Forest, XGBoost, LightGBM, and Isolation Forest Anomaly Detection

### Architectural Innovations:
1. **Tri-Ensemble Diversity:** Bagging (Random Forest) + Level-wise Boosting (XGBoost) + Leaf-wise Histogram Boosting (LightGBM).
2. **Depth Regularization:** Capping tree depths (max_depth=16 for RF, max_depth=6 for XGB/LightGBM) to prevent memorization of noise and suppress False Positives.
3. **Unsupervised Zero-Day Detection:** Isolation Forest trained strictly on benign baseline traffic with sigmoid probability calibration.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier
import lightgbm as lgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

DATA_DIR = Path('../data/raw')
train_df = pd.read_csv(DATA_DIR / 'UNSW_NB15_training-set.csv', low_memory=False)
test_df = pd.read_csv(DATA_DIR / 'UNSW_NB15_testing-set.csv', low_memory=False)

train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

excluded = ['id', 'label', 'attack_cat']
X_train = train_df.drop(columns=excluded)
y_train = train_df['label'].astype(int)

X_test = test_df.drop(columns=excluded)
y_test = test_df['label'].astype(int)

categorical_cols = ['proto', 'service', 'state']
numerical_cols = [c for c in X_train.columns if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numerical_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)


### 1. Train Individual Ensemble Models
Training Random Forest, XGBoost, and LightGBM with depth constraints.

In [2]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1, verbose=-1)
}

trained_models = {}
eval_results = []

for name, model in models.items():
    start = time.time()
    model.fit(X_train_proc, y_train)
    elapsed = round(time.time() - start, 2)
    trained_models[name] = model
    
    probs = model.predict_proba(X_test_proc)[:, 1]
    preds = (probs >= 0.5).astype(int)
    
    eval_results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall': round(recall_score(y_test, preds), 4),
        'F1-Score': round(f1_score(y_test, preds), 4),
        'ROC-AUC': round(roc_auc_score(y_test, probs), 4),
        'Train Time (s)': elapsed
    })
    print(f'{name} trained in {elapsed}s.')

display(pd.DataFrame(eval_results))


Random Forest trained in 6.42s.
XGBoost trained in 1.38s.
LightGBM trained in 1.09s.


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Train Time (s)
0,Random Forest,0.8655,0.8093,0.9887,0.8900,0.9828,6.42
1,XGBoost,0.8764,0.8235,0.9869,0.8979,0.9852,1.38
2,LightGBM,0.8730,0.8192,0.9871,0.8954,0.9851,1.09


### 2. Tri-Ensemble Soft-Voting Aggregation
Combine probabilities: {ens} = \\frac{P_{RF} + P_{XGB} + P_{LGBM}}{3}$.

In [3]:
p_rf = trained_models['Random Forest'].predict_proba(X_test_proc)[:, 1]
p_xgb = trained_models['XGBoost'].predict_proba(X_test_proc)[:, 1]
p_lgb = trained_models['LightGBM'].predict_proba(X_test_proc)[:, 1]

# Tri-ensemble soft vote
p_ens = (p_rf + p_xgb + p_lgb) / 3.0
pred_ens = (p_ens >= 0.5).astype(int)

print('=== Tri-Ensemble (RF + XGB + LightGBM) Test Performance ===')
print(f'Accuracy:  {accuracy_score(y_test, pred_ens):.4f}')
print(f'Precision: {precision_score(y_test, pred_ens):.4f}')
print(f'Recall:    {recall_score(y_test, pred_ens):.4f}')
print(f'F1-Score:  {f1_score(y_test, pred_ens):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, p_ens):.4f}')


=== Tri-Ensemble (RF + XGB + LightGBM) Test Performance ===
Accuracy:  0.8708
Precision: 0.8160
Recall:    0.9882
F1-Score:  0.8939
ROC-AUC:   0.9850


### 3. Isolation Forest for Zero-Day Anomaly Detection
Train exclusively on normal background traffic (=0$) and apply sigmoid score calibration.

In [4]:
# Normal traffic only baseline
X_normal = X_train[y_train == 0]
iso_scaler = StandardScaler()
X_normal_s = iso_scaler.fit_transform(X_normal[numerical_cols])
X_test_iso_s = iso_scaler.transform(X_test[numerical_cols])

iso_model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42, n_jobs=-1)
iso_model.fit(X_normal_s)

# Sigmoid calibration: maps raw decision function into 0.0 - 1.0 probability
raw_decision = iso_model.decision_function(X_test_iso_s)
p_iso = 1.0 / (1.0 + np.exp(6.0 * raw_decision))

print('Isolation Forest Anomaly Detector calibrated on Normal Baseline.')
print(f'Zero-Day Anomaly ROC-AUC: {roc_auc_score(y_test, p_iso):.4f}')


Isolation Forest Anomaly Detector calibrated on Normal Baseline.
Zero-Day Anomaly ROC-AUC: 0.7705
